In [ ]:
import pandas as pd
# 1
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
# 2
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
# 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
# 4
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
# 5
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])

Ваши ответы и объяснения:
1. 20 10 — по метке b получаем 20, по позиции 0 получаем 10.
2. Везде NaN, кроме Шымкента (3) — Series складываются по названиям городов.
3. (2, 2) — остаются две строки и два столбца.
4. Series DataFrame — один столбец в одиночных скобках это Series, в двойных — DataFrame.
5. [1, 2] 5 — после фильтра индексы сохраняются, а iloc выбирает по позиции.

In [1]:
# Загрузите orders.csv в Colab через панель Files перед запуском
import pandas as pd
import numpy as np

# Проверяем предсказания после ответов
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print("1:", s["b"], s.iloc[0])
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print("2:\n", a + b)
small = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print("3:", small[small["x"] > 1].shape)
print("4:", type(small["x"]).__name__, type(small[["x"]]).__name__)
f = small[small["x"] > 1]
print("5:", f.index.tolist(), f.iloc[0, 1])

1: 20 10
2:
 Алматы     NaN
Астана     NaN
Шымкент    3.0
dtype: float64
3: (2, 2)
4: Series DataFrame
5: [1, 2] 5


In [6]:
import os
print(os.getcwd())
print(os.listdir())

/content
['.config', 'sample_data']


In [12]:
df = pd.read_csv("orders.csv")

df["amount"] = pd.to_numeric(
    df["amount"].astype(str).str.replace(" ", ""),
    errors="coerce"
)

print(df["amount"].dtype)

float64


In [13]:
sep = df["order_date"] < "2026-10"
octo = ~sep
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"
SH = ["Шымкент", "шымкент", "Shymkent"]
print("Сентябрь:", sep.sum(), "Октябрь:", octo.sum())

Сентябрь: 2782 Октябрь: 2681


In [14]:
# База 1
print("Строки и столбцы:", df.shape)
print("Типы:\n", df.dtypes)
print("Пропуски delivery_days:", df["delivery_days"].isna().sum())
for col in ["city", "category", "status", "customer_id"]:
    print(col, "уникальных:", df[col].nunique())
print("Менеджеру: есть пропуски в доставке, разные написания городов и заказы TEST.")

Строки и столбцы: (5463, 8)
Типы:
 order_id           int64
order_date        object
customer_id       object
city              object
category          object
amount           float64
delivery_days    float64
status            object
dtype: object
Пропуски delivery_days: 160
city уникальных: 14
category уникальных: 5
status уникальных: 2
customer_id уникальных: 2003
Менеджеру: есть пропуски в доставке, разные написания городов и заказы TEST.


In [15]:
# База 2
x = df.loc[paid & not_test, "amount"]
print("Среднее:", round(x.mean(), 2))
print("Медиана:", x.median())
print("Q1:", x.quantile(0.25))
print("Q3:", x.quantile(0.75))
print("Менеджеру: медиана лучше показывает типичный чек, потому что большие заказы поднимают среднее.")

Среднее: 14672.39
Медиана: 10300.0
Q1: 5880.0
Q3: 18690.0
Менеджеру: медиана лучше показывает типичный чек, потому что большие заказы поднимают среднее.


In [16]:
# База 3
cancel = df["status"] == "отменён"
AL = ["Алматы", "алматы", "Almaty", " Алматы ", " Алматы", "АЛМАТЫ"]
sh = df["city"].isin(SH)
al = df["city"].isin(AL)
all_rate = cancel.mean() * 100
sh_rate = cancel[sh].mean() * 100
al_rate = cancel[al].mean() * 100
print("Все:", round(all_rate, 1), "%")
print("Шымкент:", round(sh_rate, 1), "%")
print("Алматы:", round(al_rate, 1), "%")
print("Во сколько раз:", round(sh_rate / al_rate, 1))
print("Менеджеру: в Шымкенте отмен заметно больше, чем в Алматы.")

Все: 10.2 %
Шымкент: 20.4 %
Алматы: 7.9 %
Во сколько раз: 2.6
Менеджеру: в Шымкенте отмен заметно больше, чем в Алматы.


In [17]:
# Стандарт 4
for name, variants in [("Шымкент", SH), ("Алматы", AL)]:
    city_mask = df["city"].isin(variants) & paid & not_test
    september = df.loc[city_mask & sep, "amount"].sum()
    october = df.loc[city_mask & octo, "amount"].sum()
    change = (october - september) / september * 100
    print(name, "сентябрь:", september, "октябрь:", october, "изменение:", round(change, 1), "%")
print("Менеджеру: сравниваем изменения в процентах, а не только суммы.")

Шымкент сентябрь: 4949800.0 октябрь: 4044010.0 изменение: -18.3 %
Алматы сентябрь: 15523940.0 октябрь: 14643020.0 изменение: -5.7 %
Менеджеру: сравниваем изменения в процентах, а не только суммы.


In [19]:
# Стандарт 5
cancel_sh = df[
    (df["city"].isin(["Шымкент", "шымкент", "Shymkent"])) &
    (df["status"] == "отменён")
]

print("Отменённых заказов в Шымкенте:", len(cancel_sh))
print("Вывод: отменённые заказы не должны входить в оплаченную выручку.")

Отменённых заказов в Шымкенте: 164
Вывод: отменённые заказы не должны входить в оплаченную выручку.


In [20]:
# Стандарт 6
counts = df["customer_id"].value_counts()
client = counts.index[0]
print("Самый частый клиент:", client, "заказов:", counts.iloc[0])
print(df.loc[df["customer_id"] == client, ["order_date", "city", "amount", "status"]])
print("Повторяющиеся order_id:")
print(df.loc[df["order_id"].duplicated(keep=False), "order_id"].value_counts().head())
print("Менеджеру: повторяющиеся order_id надо проверить перед итоговым отчётом.")

Самый частый клиент: C01818 заказов: 11
            order_date       city   amount   status
57    2026-09-01 15:00    Шымкент   7350.0  оплачен
779   2026-09-09 18:00    Шымкент  45270.0  оплачен
1371  2026-09-15 21:00     Алматы   8890.0  оплачен
1375  2026-09-15 21:00     Алматы   8890.0  оплачен
2195  2026-09-24 18:00     Астана   1490.0  оплачен
2617  2026-09-29 09:00     Астана  22170.0  оплачен
2776  2026-09-30 21:00     Алматы  20180.0  оплачен
3417  2026-10-08 14:00  Караганда   4330.0  оплачен
4279  2026-10-18 08:00     Астана  13600.0  оплачен
4610  2026-10-21 19:00     Алматы  21990.0  оплачен
4778  2026-10-23 18:00     Алматы  11140.0  оплачен
Повторяющиеся order_id:
order_id
102564    2
100113    2
101761    2
101188    2
101182    2
Name: count, dtype: int64
Менеджеру: повторяющиеся order_id надо проверить перед итоговым отчётом.


In [21]:
# Челлендж 7
x = df.loc[paid & not_test, "amount"]
q1 = x.quantile(0.25)
q3 = x.quantile(0.75)
iqr = q3 - q1
limit = q3 + 1.5 * iqr
out = df.loc[paid & not_test & (df["amount"] > limit)]
print("Граница:", limit)
print("Выбросов:", len(out))
print("Доля:", round(len(out) / len(x) * 100, 1), "%")
print("Самая частая категория:", out["category"].value_counts().index[0])
print("Менеджеру: IQR и z-оценка используют разные правила, поэтому количество выбросов отличается.")

Граница: 37905.0
Выбросов: 320
Доля: 6.5 %
Самая частая категория: Электроника
Менеджеру: IQR и z-оценка используют разные правила, поэтому количество выбросов отличается.


In [22]:
# Челлендж 8
all_cities = df["city"].value_counts()
cancel_cities = df.loc[df["status"] == "отменён", "city"].value_counts()
rate = (cancel_cities / all_cities).sort_values(ascending=False)
print("Доля отмен по написаниям города:\n", (rate * 100).round(1))
print("Шымкент:", all_cities.get("Шымкент", 0))
print("шымкент:", all_cities.get("шымкент", 0))
print("Shymkent:", all_cities.get("Shymkent", 0))
print("Менеджеру: по одному написанию нельзя судить о городе. Сначала нужно объединить варианты названия.")

Доля отмен по написаниям города:
 city
Shymkent     23.1
Шымкент      21.5
алматы       14.1
астана       13.1
Актобе       10.3
Astana       10.0
Караганда     8.9
Астана        8.5
Алматы        8.0
Атырау        7.5
Almaty        5.8
 Алматы       4.5
шымкент       3.7
АЛМАТЫ        3.2
Name: count, dtype: float64
Шымкент: 711
шымкент: 54
Shymkent: 39
Менеджеру: по одному написанию нельзя судить о городе. Сначала нужно объединить варианты названия.
